In [2]:
import torch
from l8_9_sncoder import EncoderL8
from s2_encoder import S2Encoder



# dummy_l8 = torch.randn(1, 11, 224, 224)
# dummy_s2 = torch.randn(1, 13, 224, 224)

# l8_encoder = EncoderL8(in_channels=11)
# s2_encoder = S2Encoder(in_channels=13)

# z_l8 = l8_encoder(dummy_l8)
# z_s2 = s2_encoder(dummy_s2)

# print(z_l8.shape, z_s2.shape)

/opt/anaconda3/envs/torch-env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [22]:
import re
import numpy as np
import rasterio
from rasterio.enums import Resampling
from pathlib import Path

def read_safe_l1c(path, stack=True, resample=False, bands=None):
    path = Path(path)
    default_bands = ["B01","B02","B03","B04","B05","B06","B07",
                     "B08","B8A","B09","B10","B11","B12"]

    if bands is None:
        patterns = default_bands
    else:
        patterns = [f"B{str(b).zfill(2)}" if isinstance(b,int) else str(b).upper() for b in bands]

    regexes = [re.compile(p, flags=re.IGNORECASE) for p in patterns]

    img_dirs = [d for d in path.rglob("*") if d.is_dir() and "IMG_DATA" in d.name]
    if not img_dirs:
        raise ValueError(f"No IMG_DATA found in {path}")
    img_dir = img_dirs[0]

    files = [p for p in img_dir.glob("*.jp2") if p.is_file() and "TCI" not in p.name.upper()]
    valid_files = [f for f in files if any(r.search(f.name) for r in regexes)]
    if not valid_files:
        raise ValueError(f"No matching bands found for {patterns}")

    arrays = {}
    ref_profile, meta = None, None
    for f in valid_files:
        fname = f.name.upper()
        if "B8A" in fname:
            band_id = "B8A"
        else:
            m = re.search(r"B(\d+)", fname)
            if not m:
                continue
            band_id = f"B{m.group(1).zfill(2)}"

        with rasterio.open(f) as src:
            if meta is None:
                meta = src.meta.copy()
            if resample and ref_profile is not None:
                arr = src.read(1, out_shape=(ref_profile['height'], ref_profile['width']),
                               resampling=Resampling.cubic)
            else:
                arr = src.read(1)
                ref_profile = src.profile
            arrays[band_id] = arr

    band_order = patterns if bands is not None else default_bands
    missing = [b for b in band_order if b not in arrays]
    if missing:
        raise ValueError(f"Missing bands: {missing}")

    data = np.stack([arrays[b] for b in band_order], axis=0) if stack else {b: arrays[b] for b in band_order}
    meta.update(ref_profile)
    return data, meta


In [23]:
data, meta = read_safe_l1c("/Users/elqajjammohammed/Desktop/Extra-Work/PoCs/SatRing-One-Embedding-to-Bind-Them-all/src/encoders/test_data/S2A_MSIL1C_20240601T105621_N0510_R094_T30SUC_20240601T162135.SAFE", stack=True, resample=True)


In [24]:
s2_encoder = S2Encoder(in_channels=13)


In [ ]:

x = torch.from_numpy(data).unsqueeze(0).float()
x /= 10000.0  # Normalize to [0, 1] range if needed
z_s2 = s2_encoder(x)


In [26]:
z_s2

tensor([[0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0096, 0.0000]],
       grad_fn=<ViewBackward0>)